In [ ]:
import os
import json
import pandas as pd

In [ ]:
######################## data 불러오기 ##########################################################################

In [ ]:
file_path = r"./024.아동·청소년 상담 데이터\3.개방데이터\1.데이터\Training\02.라벨링데이터\TL_out"

In [ ]:
files = os.listdir(file_path)

In [ ]:
json_data_list = []
for file in files:
    if file.endswith(".json"):  # JSON 파일만 선택
        with open(os.path.join(file_path, file), 'r', encoding='utf-8') as f:
            json_data = json.load(f)  # JSON 파일 읽기
            json_data_list.append(json_data)

In [ ]:
len(json_data_list)

In [ ]:
######################## 1. 학대여부 대화문 추출 ###################################################################

In [ ]:
########### 1.1 질문 + 응답

In [ ]:
def extract_dialogues(data):
    result = []
    
    # '문항'이 '학대여부'인 부분만 필터링
    for entry in data.get('list', []):
        if entry.get('문항') == '학대여부':
            row = {}
            for 항목 in entry.get('list', []):
                항목_name = 항목.get('항목')  # 방임, 정서학대, 신체학대, 성학대
                
                # Q & A 텍스트 추출
                dialogues = []
                for audio in 항목.get('audio', []):
                    type_prefix = "Q" if audio['type'] == 'Q' else "A"
                    text = f"{type_prefix}: {audio['text']}"
                    dialogues.append(text)
                
                # 대화문을 항목 이름으로 데이터에 추가
                row[항목_name] = " ".join(dialogues)  # 대화문을 하나의 문자열로 결합
                
            result.append(row)
    
    return result

In [ ]:
all_rows = []
for json_data in json_data_list:
    dialogues = extract_dialogues(json_data)
    all_rows.extend(dialogues)

In [ ]:
df_QA = pd.DataFrame(all_rows)

In [ ]:
print(df_QA.head)

In [ ]:
print(f"DataFrame 차원: {df_QA.shape}")

In [ ]:
df_QA.iloc[0,0]

In [ ]:
########### 1.2 응답만

In [ ]:
def extract_answers_only(data):
    result = []
    
    # '문항'이 '학대여부'인 부분만 필터링
    for entry in data.get('list', []):
        if entry.get('문항') == '학대여부':
            row = {}
            for 항목 in entry.get('list', []):
                항목_name = 항목.get('항목')  # 방임, 정서학대, 신체학대, 성학대
                
                # A 타입 (답변) 텍스트만 추출
                answers = []
                for audio in 항목.get('audio', []):
                    if audio['type'] == 'A':  # A 타입만 선택
                        answers.append(audio['text'])
                
                # 답변 텍스트를 항목 이름으로 데이터에 추가
                row[항목_name] = " ".join(answers)  # 답변을 하나의 문자열로 결합
                
            result.append(row)
    
    return result

In [ ]:
all_rows = []
for json_data in json_data_list:
    answers = extract_answers_only(json_data)
    all_rows.extend(answers)

In [ ]:
df_A = pd.DataFrame(all_rows)

In [ ]:
df_A.iloc[0,0]

In [ ]:
######################## 2. 나머지 정보 추출 ###################################################################

In [ ]:
def extract_combined_info(data):
    row = {}
    
    # 1. 'info'에서 객체 정보 추출
    info = data.get('info', {})
    row['성별'] = info.get('성별', '')
    row['나이'] = info.get('나이', '')
    row['학년'] = info.get('학년', '')
    row['유형구분'] = info.get('유형구분', '')
    row['위기단계'] = info.get('위기단계', '')
    row['합계점수'] = info.get('합계점수', 0)
    row['학대의심'] = info.get('학대의심', '')
    row['행동특성 점수'] = info.get('행동특성 점수', 0)
    
    # 2. 'list'에서 학대여부를 제외한 각 항목별 대화 추출
    for entry in data.get('list', []):
        if entry.get('문항') != '학대여부':  # '학대여부' 문항을 제외
            for 항목 in entry.get('list', []):
                항목_name = 항목.get('항목', '')  # 각 항목 이름 (예: 통증, 신체손상 등)

                # Q & A 텍스트 추출
                dialogues = []
                if 'audio' in 항목:  # 'audio' 항목이 존재하는지 확인
                    for audio in 항목.get('audio', []):
                        type_prefix = "Q" if audio.get('type', '') == 'Q' else "A"
                        text = f"{type_prefix}: {audio.get('text', '')}"  # 텍스트가 없을 경우 빈 문자열 처리
                        dialogues.append(text)

                # 대화가 있는 경우 항목 이름을 열로 하여 추가
                row[항목_name] = " ".join(dialogues) if dialogues else ''  # 대화가 없을 경우 빈 문자열

    return row

In [ ]:
all_rows = []
for idx, json_data in enumerate(json_data_list):
    dialogues = extract_combined_info(json_data)
    all_rows.append(dialogues)  

In [ ]:
df_other = pd.DataFrame(all_rows)

In [ ]:
print(df_other.head())

In [ ]:
print("열 이름:", df_other.columns.tolist())